In [1]:
import cv2
import numpy as np

# Load images (use FULL PATH)
ref_img = cv2.imread("C:/Users/lenovo/Desktop/IPCV/images/reference.jpg", 0)
scene_img = cv2.imread("C:/Users/lenovo/Desktop/IPCV/images/scene.jpg")
overlay_img = cv2.imread("C:/Users/lenovo/Desktop/IPCV/images/overlay.png")

# Check images
if ref_img is None or scene_img is None or overlay_img is None:
    print("❌ Error: Check image paths")
    exit()

# Resize overlay to reference size
overlay_img = cv2.resize(overlay_img, (ref_img.shape[1], ref_img.shape[0]))

# ORB detector
orb = cv2.ORB_create(nfeatures=1000)

# Detect keypoints & descriptors
kp1, des1 = orb.detectAndCompute(ref_img, None)
kp2, des2 = orb.detectAndCompute(cv2.cvtColor(scene_img, cv2.COLOR_BGR2GRAY), None)

# Matcher
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

# Sort matches
matches = sorted(matches, key=lambda x: x.distance)

# Take good matches
good_matches = matches[:50]

# Check minimum matches
if len(good_matches) < 10:
    print("❌ Not enough matches found")
    exit()

# Extract points
src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)

# Compute homography
H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

# Warp overlay image
overlay_warp = cv2.warpPerspective(
    overlay_img,
    H,
    (scene_img.shape[1], scene_img.shape[0])
)

# 🔥 FIX: Proper masking
gray_overlay = cv2.cvtColor(overlay_warp, cv2.COLOR_BGR2GRAY)
_, mask_overlay = cv2.threshold(gray_overlay, 1, 255, cv2.THRESH_BINARY)

mask_inv = cv2.bitwise_not(mask_overlay)

# Remove area from scene
scene_bg = cv2.bitwise_and(scene_img, scene_img, mask=mask_inv)

# Extract overlay part
overlay_fg = cv2.bitwise_and(overlay_warp, overlay_warp, mask=mask_overlay)

# Combine
result = cv2.add(scene_bg, overlay_fg)

# Show result
cv2.imshow("Augmented Reality Result", result)
cv2.waitKey(0)
cv2.destroyAllWindows()